<a href="https://colab.research.google.com/github/Sasitorn-ann/ai_lab/blob/main/LabWeek5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
x = 8.0
lr = 0.1
for i in range(21):
    if i in (0, 1, 2, 3, 5, 10, 20):
        print(f"ก้าว {i:2d}  x = {x:7.4f}  ความสูง = {x*x:8.4f}")
    gradient = 2*x            # ความชันใต้ฝ่าเท้า
    x = x - lr*gradient       # ก้าวสวนทางความชัน

ก้าว  0  x =  8.0000  ความสูง =  64.0000
ก้าว  1  x =  6.4000  ความสูง =  40.9600
ก้าว  2  x =  5.1200  ความสูง =  26.2144
ก้าว  3  x =  4.0960  ความสูง =  16.7772
ก้าว  5  x =  2.6214  ความสูง =   6.8719
ก้าว 10  x =  0.8590  ความสูง =   0.7379
ก้าว 20  x =  0.0922  ความสูง =   0.0085


In [5]:
for lr in [0.01, 0.1, 0.45, 0.55, 1.05]:
    x = 8.0
    for _ in range(20):
        x = x - lr*2*x
    print(f"lr {lr:5.2f}  หลัง 20 ก้าว x = {x:.4f}")

lr  0.01  หลัง 20 ก้าว x = 5.3409
lr  0.10  หลัง 20 ก้าว x = 0.0922
lr  0.45  หลัง 20 ก้าว x = 0.0000
lr  0.55  หลัง 20 ก้าว x = 0.0000
lr  1.05  หลัง 20 ก้าว x = 53.8200


In [15]:
import numpy as np
rng = np.random.default_rng(1)
X = np.array([[0,0],[0,1],[1,0],[1,1]], float)
y = np.array([[0],[1],[1],[0]], float)

In [22]:
X

array([[0., 0.],
       [0., 1.],
       [1., 0.],
       [1., 1.]])

In [25]:
y

array([[0.],
       [1.],
       [1.],
       [0.]])

In [9]:
W1 = rng.normal(0, 1, (2,2)); b1 = np.zeros((1,2))
W2 = rng.normal(0, 1, (2,1)); b2 = np.zeros((1,1))
sig = lambda z: 1/(1+np.exp(-z))

In [17]:
W1

array([[-6.0799352 ,  6.45411964],
       [ 5.89209501, -6.65526334]])

In [18]:
W2

array([[10.20104353],
       [10.03619005]])

In [19]:
h = sig(X@W1 + b1)            # ชั้นซ่อนคิด
out = sig(h@W2 + b2)          # ชั้นตอบคิด

In [20]:
print("คำตอบก่อนฝึก", np.round(out.ravel(), 3))
print("loss ก่อนฝึก", round(float(np.mean((out-y)**2)), 4))

คำตอบก่อนฝึก [0.013 0.989 0.989 0.011]
loss ก่อนฝึก 0.0001


In [11]:
lr = 0.5
for epoch in range(20001):
    h = sig(X@W1 + b1)                  # จังหวะ 1 คิดไปข้างหน้า
    out = sig(h@W2 + b2)
    loss = np.mean((out-y)**2)          # จังหวะ 2 วัดความผิด
    if epoch in (0, 100, 1000, 5000, 20000):
        print(f"รอบ {epoch:5d}  loss = {loss:.4f}")

    d_out = (out-y)*out*(1-out)         # จังหวะ 3 ใบตำหนิชั้นตอบ
    d_h = d_out@W2.T * h*(1-h)          # ส่งย้อนไปชั้นซ่อน
    W2 -= lr*h.T@d_out; b2 -= lr*d_out.sum(0)
    W1 -= lr*X.T@d_h;  b1 -= lr*d_h.sum(0)

print("คำตอบหลังฝึก", np.round(out.ravel(), 3))

รอบ     0  loss = 0.2799
รอบ   100  loss = 0.2486
รอบ  1000  loss = 0.0155
รอบ  5000  loss = 0.0007
รอบ 20000  loss = 0.0001
คำตอบหลังฝึก [0.013 0.989 0.989 0.011]


In [12]:
print(np.round(sig(X@W1 + b1), 2))

[[0.04 0.03]
 [0.93 0.  ]
 [0.   0.95]
 [0.03 0.02]]


In [34]:
import torch
import torch.nn as nn

In [35]:
torch.manual_seed(1)
X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y = torch.tensor([[0.],[1.],[1.],[0.]])

In [36]:
model = nn.Sequential(nn.Linear(2,2), nn.Sigmoid(),
                      nn.Linear(2,1), nn.Sigmoid())
opt = torch.optim.SGD(model.parameters(), lr=0.5)
loss_fn = nn.MSELoss()

In [13]:
for epoch in range(20001):
    out = model(X)                      # คิดไปข้างหน้า
    loss = loss_fn(out, y)              # วัดความผิด
    if epoch in (0, 100, 1000, 5000, 20000):
        print(f"รอบ {epoch:5d}  loss = {loss.item():.4f}")
    opt.zero_grad()
    loss.backward()                     # ใบตำหนิย้อนกลับ ในบรรทัดเดียว
    opt.step()                          # ขยับทุกน้ำหนักหนึ่งก้าว

print("คำตอบหลังฝึก", model(X).detach().numpy().round(3).ravel())

รอบ     0  loss = 0.2548
รอบ   100  loss = 0.2501
รอบ  1000  loss = 0.2500
รอบ  5000  loss = 0.0355
รอบ 20000  loss = 0.0004
คำตอบหลังฝึก [0.021 0.982 0.982 0.019]


In [14]:
def train_xor(lr, epochs=20000, seed=1):
    rng = np.random.default_rng(seed)
    X = np.array([[0,0],[0,1],[1,0],[1,1]], float)
    y = np.array([[0],[1],[1],[0]], float)
    W1 = rng.normal(0,1,(2,2)); b1 = np.zeros((1,2))
    W2 = rng.normal(0,1,(2,1)); b2 = np.zeros((1,1))
    sig = lambda z: 1/(1+np.exp(-z))
    for _ in range(epochs):
        h = sig(X@W1+b1); out = sig(h@W2+b2)
        d_out = (out-y)*out*(1-out)
        d_h = d_out@W2.T * h*(1-h)
        W2 -= lr*h.T@d_out; b2 -= lr*d_out.sum(0)
        W1 -= lr*X.T@d_h;  b1 -= lr*d_h.sum(0)
    h = sig(X@W1+b1); out = sig(h@W2+b2)
    return float(np.mean((out-y)**2)), np.round(out.ravel(), 3)

for lr in [0.01, 0.5, 5.0, 20.0]:
    L, preds = train_xor(lr)
    print(f"lr {lr:5.2f}  loss = {L:.4f}  คำตอบ = {preds}")

lr  0.01  loss = 0.2157  คำตอบ = [0.441 0.404 0.673 0.454]
lr  0.50  loss = 0.0001  คำตอบ = [0.013 0.989 0.989 0.011]
lr  5.00  loss = 0.0000  คำตอบ = [0.004 0.997 0.997 0.003]
lr 20.00  loss = 0.1443  คำตอบ = [0.    0.303 0.998 0.303]


ด่านที่ 8
1. สามจังหวะของการฝึกเครือข่ายประสาทคืออะไร และในโค้ด PyTorch ของด่านที่ 6 แต่ละจังหวะอยู่บรรทัดไหน

Ans. บรรทัดที่ 10 out = model(X)
     บรรทัดที่ 11 loss = loss_fn(out,y)
     บรรทัดที่ 15 loss.backward()
     บรรทัดที่ 16 opt.step()

2. โมเดลภาษาใหญ่ฝึกด้วยกลไกเดียวกับสมองจิ๋วของเรา จากสิ่งที่เห็นในแล็บนี้ อธิบายว่าทำไมโมเดลที่ฝึกแบบนี้ถึงตอบเก่งมาก แต่ก็ตอบผิดอย่างมั่นใจได้ด้วย (บอกใบ้ ตลอดการฝึก เราถามมันว่าอะไร และไม่เคยถามว่าอะไร)

Ans. ตอบเก่ง :เพราะฝึกปรับน้ำหนักซ้ำเพื่อทำให้ค่า loss ต่ำที่สุด

ตอบผิดอย่างมั่นใจ:เพราะถ้าเราถามเเค่"ค่าคงที่คำนวณได้ตรงกับเป้าหมายไหม"เเต่ไม่เคยถามว่าถูกต้องตรงกับความจริงในโลกไหม

3. ถ้าฝึกโมเดลแล้ว loss พุ่งขึ้นเรื่อยๆ แทนที่จะลด สาเหตุแรกที่ควรสงสัยคืออะไร และจะลองแก้ยังไง

Ans. สาเหตุ Learning rate สูงเกินไป เเก้ด้วยการลด lr ให้เล็กลง





#ตอบคำถามชวนคิดด่านที่ 2
ก้าว 0.55 น่าสนใจที่สุด มันก้าวข้ามก้นหุบไปตกฝั่งตรงข้ามทุกก้าว (ลอง print ค่า x ทุกก้าวดู จะเห็นเครื่องหมายบวกลบสลับกัน) แต่สุดท้ายก็ถึงก้นหุบ ทำไมมันต่างจาก 1.05 ที่เด้งเหมือนกันแต่พัง (บอกใบ้ เทียบขนาดของ x แต่ละรอบว่าหดลงหรือโตขึ้น)

Ans. ต่างกันที่ขนาดตัวเลข X ในเเต่ละรอบว่ามันหดลงหรือโตขึ้น

In [33]:
for lr in [0.01, 0.1, 0.45, 0.55, 1.05]:
    x = 8.0
    print(f"\nlr = {lr}")
    for step in range(20):
        x = x - lr*2*x
        print(f"ก้าว {step+1:2d}: x = {x:.6f}")


lr = 0.01
ก้าว  1: x = 7.840000
ก้าว  2: x = 7.683200
ก้าว  3: x = 7.529536
ก้าว  4: x = 7.378945
ก้าว  5: x = 7.231366
ก้าว  6: x = 7.086739
ก้าว  7: x = 6.945004
ก้าว  8: x = 6.806104
ก้าว  9: x = 6.669982
ก้าว 10: x = 6.536582
ก้าว 11: x = 6.405851
ก้าว 12: x = 6.277734
ก้าว 13: x = 6.152179
ก้าว 14: x = 6.029136
ก้าว 15: x = 5.908553
ก้าว 16: x = 5.790382
ก้าว 17: x = 5.674574
ก้าว 18: x = 5.561083
ก้าว 19: x = 5.449861
ก้าว 20: x = 5.340864

lr = 0.1
ก้าว  1: x = 6.400000
ก้าว  2: x = 5.120000
ก้าว  3: x = 4.096000
ก้าว  4: x = 3.276800
ก้าว  5: x = 2.621440
ก้าว  6: x = 2.097152
ก้าว  7: x = 1.677722
ก้าว  8: x = 1.342177
ก้าว  9: x = 1.073742
ก้าว 10: x = 0.858993
ก้าว 11: x = 0.687195
ก้าว 12: x = 0.549756
ก้าว 13: x = 0.439805
ก้าว 14: x = 0.351844
ก้าว 15: x = 0.281475
ก้าว 16: x = 0.225180
ก้าว 17: x = 0.180144
ก้าว 18: x = 0.144115
ก้าว 19: x = 0.115292
ก้าว 20: x = 0.092234

lr = 0.45
ก้าว  1: x = 0.800000
ก้าว  2: x = 0.080000
ก้าว  3: x = 0.008000
ก้าว  4: x = 0.000800
